# 02 — Statistical framework (experiment design, ablations)

**Variant:** `typewell_gr_alignment` · Reads phase 01 manifest · Writes `artifacts/02_statistical_framework/`

Documents Ke (2017) LightGBM + Pedregosa (2011) ColumnTransformer factorial design, CV metric, and ADR — at the narrative depth of top Rogii solution writeups.


In [ ]:
"""Shared imports for Rogii TVT pipeline notebooks."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
if not (NB_DIR / "phase_runner.py").is_file():
    NB_DIR = Path(r"/lustre/work/sweeden/agent-tracing-trace-baseline/examples/rogii/traces/preprocessing/typewell_gr_alignment/notebooks")
VARIANT_DIR = NB_DIR.parent
ROGII_ROOT = Path("/lustre/work/sweeden/rogii")
sys.path.insert(0, str(NB_DIR))
if str(ROGII_ROOT) not in sys.path:
    sys.path.insert(0, str(ROGII_ROOT))

from phase_runner import (
    ARTIFACTS_ROOT,
    TRACE_INDEX,
    require_prior_phase,
    load_phase_manifest,
    trace_steps_for_phase,
    _resolve_path,
    _read_json,
    _load_train_predict,
)

pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid")

PHASE = "02_statistical_framework"


## 1. Verify phase 01 handoff


In [ ]:
p01 = require_prior_phase(PHASE)
print("prior:", p01["phase"], p01["completed_at"])
well_groups = _read_json(_resolve_path(p01["outputs"]["well_groups"]))
target_diag = _read_json(_resolve_path(p01["outputs"]["target_diagnosis"]))
eda_summary = _read_json(_resolve_path(p01["outputs"]["eda_summary"]))
print("group_key:", well_groups.get("group_key"))
print("use_log1p:", target_diag.get("use_log1p"))


## 2. Load experiment descriptor and base papers


In [ ]:
desc = json.loads((VARIANT_DIR / "experiment_descriptor.json").read_text())
abl = json.loads((VARIANT_DIR / "ablation_plan.json").read_text())
mle = json.loads((VARIANT_DIR / "mle_plan.json").read_text())
print("hypothesis:", desc["experiment"]["hypothesis"][:120], "...")
print("base paper:", desc["base_paper"]["title"])
print("ablation factors:", [f["id"] for f in abl.get("ablation_factors", [])])


## 3. Phase 01 → design decisions

Translate EDA into CV scheme, target transform, and high-missing feature list.


In [ ]:
recommended_cv = "nested_groupkfold_by_well" if well_groups.get("group_key") else "kfold"
recommended_target_transform = "log1p" if target_diag.get("use_log1p") else "none"
high_missing = [c for c, r in eda_summary.get("missingness_rate", {}).items() if float(r) >= 0.40]
print("recommended_cv:", recommended_cv)
print("target_log1p:", recommended_target_transform)
print("high_missing_features:", high_missing)


## 4. Ablation factorial grid preview


In [ ]:
from pipeline.agents import ExperimentDesignArchitect
architect = ExperimentDesignArchitect()
factors = {f["id"]: f["levels"] for f in abl.get("ablation_factors", []) if "id" in f}
grid = architect.design_ablation_factorial_grid(factors)
print("n_runs:", len(grid))
pd.DataFrame(grid).head(12)


## 5. Training plan and episodic schedule


In [ ]:
training_plan = architect.design_factorial_and_episodic_training(factors)
print(json.dumps({k: training_plan[k] for k in list(training_plan)[:6]}, indent=2)[:1200])


## 6. Paper citations trace map


In [ ]:
citations = architect.cite_base_paper_ablations(desc)
print(json.dumps(citations, indent=2)[:1500])


## 7. Seed control and reproducibility


In [ ]:
from pipeline.agents import SeedControlOfficer
SeedControlOfficer().pin_seed(42)
print("seed pinned: 42")


## 8. ADR — architecture decision record


In [ ]:
from pipeline.agents import ADRRecorder
adr = ADRRecorder()
adr.record_initial_adr(rationale="baseline_column_transformer phase 02 — notebook walkthrough")
print(adr.serialize_markdown())


## 9. Cross-validation rationale

When `group_key` is null we use **KFold** (not GroupKFold) to avoid pseudo-groups; when wells are identifiable, **GroupKFold** prevents leakage across horizontal well tails.


In [ ]:
print("Phase 01 recommended_cv:", recommended_cv)
print("n_unique_groups:", well_groups.get("n_unique_groups"))


## 10. Target transform contract

Log1p is only enabled when skewness + positivity diagnostics from phase 01 recommend it; inverse transform uses `expm1` at inference.


In [ ]:
print(json.dumps(target_diag, indent=2))


## 11. Ablation factor level counts


In [ ]:
for fid, levels in factors.items():
    print(f"{fid}: {len(levels)} levels → {levels}")


## 12. MLE plan snapshot


In [ ]:
print(json.dumps(mle, indent=2)[:2000])


## 13. Factorial grid heatmap (target_log1p × scaler)


In [ ]:
gdf = pd.DataFrame(grid)
if {"target_log1p", "numeric_scaler"}.issubset(gdf.columns):
    pivot = gdf.groupby(["target_log1p", "numeric_scaler"]).size().unstack(fill_value=0)
    display(pivot)


## 14. SMRE and metric contract


In [ ]:
print("metric:", desc["experiment"].get("smre"), "| hypothesis metric: rmse_post_ps")


## 15. Persist phase 02 artifacts


In [ ]:
from phase_runner import run_02_statistical_framework
manifest = run_02_statistical_framework()
print(json.dumps(manifest, indent=2))


## 16. Review statistical_framework.json


In [ ]:
sf = _read_json(ARTIFACTS_ROOT / PHASE / "statistical_framework.json")
print(json.dumps(sf, indent=2))


## 17. Ablation grid summary table


In [ ]:
grid_doc = _read_json(ARTIFACTS_ROOT / PHASE / "ablation_grid.json")
pd.DataFrame(grid_doc["grid"]).describe(include="all")


## 18. Trace step coverage


In [ ]:
steps = trace_steps_for_phase(PHASE)
print(f"Trace steps executed for {PHASE}: {len(steps)}")
pd.DataFrame(steps)[["trace_row", "agent", "token"]].head(15)


## 19. Phase 02 completion checklist


In [ ]:
checks = {
    "statistical_framework.json": (ARTIFACTS_ROOT / PHASE / "statistical_framework.json").is_file(),
    "ablation_grid.json": (ARTIFACTS_ROOT / PHASE / "ablation_grid.json").is_file(),
    "phase_manifest.json": (ARTIFACTS_ROOT / PHASE / "phase_manifest.json").is_file(),
}
pd.Series(checks)


## 20. Handoff to phase 03

Next notebook consumes `feature_config` + `cv_config` from phase 03 artifacts after ColumnTransformer assembly.


Proceed to `03_feature_engineering.ipynb` once this manifest shows `completed_at`.
